In [1]:
from iFinDPy import *
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL
from datetime import date
import pandas as pd
import numpy as np
import re

e:\ProgramData\Anaconda3\envs\QGIBG\Lib\site-packages\iFinDPy.pth


In [177]:

sql_host='localhost'
sql_user='Local_Editor'
sql_password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(sql_host,sql_user,sql_password,database)

06-17 14:54 logger       INFO     using username Local_Editor
06-17 14:54 logger       INFO     accessed to database <qgdbs>


In [3]:
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/qg_code_init_log.log",'a', encoding='utf-8')
    fh.setLevel(logging.CRITICAL)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    fh.setFormatter(formatter)
    logger.addHandler(ch)
    logger.addHandler(fh)
    return logger


In [4]:
logger=get_logger()

In [165]:
def qgcode_generator(mysql,type:str)->str:
    '''generate a new code for Exxx or Cxxx  qgcode'''
    #type='e'or 'c'
    sql='select qg_code from qg_indicator_info where qg_code like "%s%%.QG" order by qg_code desc limit 1'%type
    #print(sql)
    old=mysql.read_query(sql)
    if len(old)==0:
        num=1
    else:
        num=int(re.findall(r"\d+",old[0][0])[0])+1
    new=type.upper()+str(num).zfill(5)+'.QG'
    
    #mysql.close()
    
    return new



In [72]:
qgcode_generator(mysql,'C')

06-16 15:30 logger       INFO     got a cursor
06-16 15:30 logger       INFO     got a cursor
06-16 15:30 logger       INFO     Accepted read sql query "select qg_code from qg_indicator_info where qg_code like "C%.QG" order by qg_code desc limit 1"
06-16 15:30 logger       INFO     Accepted read sql query "select qg_code from qg_indicator_info where qg_code like "C%.QG" order by qg_code desc limit 1"


'C00005.QG'

In [ ]:
def add_qg_indicator(mysql,name,category,description='',need_process=1):
    #category: edb.customized.ticker 
    #need process :1=true 0=false
    if category=='edb':
        code=qgcode_generator(mysql,'e')
    elif category=='customized':
        code=qgcode_generator(mysql,'c')
    else:
        raise 
    sql='insert into qg_indicator_info (qg_code,name,category,description,need_process) value ("%s","%s","%s","%s","%s") ' %(code,name,category,description,need_process)
    
    result=mysql.write_query(sql)
    
    logger.critical(code+'      '+name)

    return code


In [ ]:
import time

In [ ]:
for t in ticker:
    #code=qgcode_generator(mysql,'e')
    sql='insert into qg_indicator_info (qg_code,name,category) value ("%s","%s","%s") ' %(t,d[d['数据代码']==t]['指标名称'].item(),'ticker')
    r=mysql.write_query(sql)
    #sql='insert into ticker_info('

In [ ]:
for t in ticker:

    sql='insert into ticker_info (code,name,region,type) value("%s","%s","%s","%s") '%(t,d[d['数据代码']==t]['指标名称'].item(),d[d['数据代码']==t]['国家'].item(),d[d['数据代码']==t]['指标类型'].item())
    r=mysql.write_query(sql)

In [ ]:
df=pd.DataFrame(data=(mysql.read_query('select code,name from ticker_info where not type="stock" ')),columns=['code','name'])
df

In [ ]:
df.to_excel('ticker.xlsx')

In [ ]:
r=mysql.read_query('select qg_code,name from edb_info where name like "美国：国债收益率%"')
r

In [ ]:
for d in r:

    sql='insert ignore into qg_indicator_info  (qg_code,name,category)  value("%s","%s","edb") '%(d[0],d[1])
    result=mysql.write_query(sql)
    print(result)

In [71]:
r=mysql.read_query('select code,name from ticker_info where code like "%.SHF"  or code like "%.CZC"')
pd.DataFrame(list(r),columns=['c','n'])

06-16 13:20 logger       INFO     got a cursor
06-16 13:20 logger       INFO     got a cursor
06-16 13:20 logger       INFO     Accepted read sql query "select code,name from ticker_info where code like "%.SHF"  or code like "%.CZC""
06-16 13:20 logger       INFO     Accepted read sql query "select code,name from ticker_info where code like "%.SHF"  or code like "%.CZC""


,c,n
0,AG.SHF,SHFE 银（主连）
1,AL.SHF,SHFE 铝（主连）
2,AP.CZC,CZCE 苹果（主连）
3,AU.SHF,SHFE 金（主连）
4,BU.SHF,SHFE 沥青（主连）
5,CF.CZC,CZCE 棉花（主连）
6,CJ.CZC,CZCE 红枣（主连）
7,CU.SHF,SHFE 铜（主连）
8,FG.CZC,CZCE 玻璃（主连）
9,FU.SHF,SHFE 燃油（主连）


In [305]:
class QGCodeLoader():
    ''' 检查一个输入的qgnode信息是否合法，合法则录入qgindicatorinfo并根据分类录入下属的三个info，并录入其他相关信息；不合法则不入库并返回错误信息。
    customized has no code before loaded'''
    '''eg input: nodeinfo={'code':'',     #required when category!=customized
                            'name':''     # always required
                            'categoty'     #required [ticker edb customized]
                            'desc'          #default None
                            'need_process'  #default 1(if not 0 ,set 1)
                            others:
                            'type'  # must have a type for ticker(like 'stock')
                            '

                            '''
    def __init__(self, **kwargs):
        #self.mysql=kwargs.get('mysql',False)

        try:
            self.name = kwargs['name']
            self.category = kwargs['category']
            if self.category == 'customized':
                mysql = MYSQL('localhost', 'Local_Editor', 'QuantumGalaxy',
                              'qgdbs')
                kwargs['qg_code'] = qgcode_generator(mysql, 'c')
                mysql.close()
            self.code = kwargs['qg_code']

            if self.category not in ['ticker', 'edb', 'customized']:
                logger.error(
                    'illegal category got, expect "ticker/edb/customized", got %s'
                    % kwargs['category'])
                return False
        except KeyError as e:
            logger.error('had error when init nodeinfo: %s not found' % e)
            return False
        self.kwargs = kwargs
        self.kwargs['description'] = kwargs.get('description', None)
        self.kwargs['need_process'] = kwargs.get('need_process', 1)
        if self.kwargs['need_process'] != 0:
            self.kwargs['need_process'] = 1
        'load node info into self.kwargs, check and initial attrs'

    def _get_mysql(self):
        sql_host = 'localhost'
        sql_user = 'Local_Editor'
        sql_password = 'QuantumGalaxy'
        database = 'qgdbs'
        mysql = MYSQL(sql_host, sql_user, sql_password, database)
        return mysql

    def mysql_write(self, sql, val=None):
        mysql = self._get_mysql()
        if val:

            mysql.wirte_many_query(sql, val)
        else:
            mysql.write_query(sql)
        mysql.close()

    def mysql_read(self, sql):
        mysql = self._get_mysql()
        r = mysql.read_query(sql)
        mysql.close()
        return r

    def check_attr_and_find_target_db(self):
        '''verify nodeinfo is legal ,set default value if not found.
        find target 2nd datebase by category, return 2nd db string'''

        if self.category == 'ticker':
            #ticker should have a type like 'stock' or other

            if not self.kwargs.get('type', False):

                logger.error('no TYPE found for ticker indicator')
                return False
            if self.kwargs.get('status', 1) != 0:
                self.kwargs['status'] = 1
            if not self.kwargs.get('summary', None):
                #set a default value for args which needed
                self.kwargs['summary'] = None
            if not self.kwargs.get('region', None):
                self.kwargs['region'] = None

            return 'ticker'
        elif self.category == 'edb':
            ths_code = self.kwargs.get('ths_code', None)
            wind_code = self.kwargs.get('wind_code', None)
            if (not ths_code and not wind_code) or (ths_code and wind_code):
                e = 'neither thscode nor windcode found; or both found two code:  '
                logger.error(e + ths_code + '  ' + wind_code)
                return False
            self.kwargs['ths_code']=re.findall(r"(\w+\w)",self.kwargs['ths_code'])[0]
            self.kwargs['wind_code']=re.findall(r"(\w+\w)",self.kwargs['wind_code'])[0]
            if not self.kwargs.get('unit', False):
                logger.error('no UNIT found for edb indicator')
                return False
            if not self.kwargs.get('summary', False):
                self.kwargs['summary'] = None
            if not self.kwargs.get('frequency', False):
                self.kwargs['frequency'] = 1

            return 'edb'
        elif self.category == 'customized':
            if not self.kwargs.get('summary', False):
                self.kwargs['summary'] = None
            return 'customized'

    def check_code(self):
        '''check if the code already in qg_indicator_info'''
        self.kwargs['qg_code'] = re.findall(r"(\w+.\w+)",self.kwargs['qg_code'])[0]
        sql = 'select name from qg_indicator_info where qg_code= "%s"' % self.kwargs[
            'qg_code']
        r = self.mysql_read(sql)
        return r

    def load(self):
        if not self.check_code():
            if self.check_attr_and_find_target_db():
                sql0 = 'insert into qg_indicator_info (qg_code,name,category,description,need_process) value("%s","%s","%s","%s","%s")' % (
                    self.kwargs.get('qg_code'), self.kwargs.get('name'),
                    self.kwargs.get('category'),
                    self.kwargs.get('description'),
                    self.kwargs.get('need_process'))

                print(sql0)
                #self.mysql_write(sql0)
                if self.category == 'ticker':
                    sql = 'insert into ticker_info (code,name,region,status,type,business) value ("%s","%s","%s","%s","%s","%s")' % (
                        self.kwargs.get('qg_code'), self.kwargs.get('name'),
                        self.kwargs.get('region'), self.kwargs.get('status'),
                        self.kwargs.get('type'), self.kwargs.get('business'))
                    print(sql)
                    #self.mysql_write(sql)
                elif self.category == 'edb':
                    sql = 'insert into edb_info (code,name,ths_code,wind_code,frequency,summary,unit) value ("%s","%s","%s","%s","%s","%s","%s") ' % (
                        self.kwargs.get('qg_code'), self.kwargs.get('name'),
                        self.kwargs.get('ths_code'),
                        self.kwargs.get('wind_code'),
                        self.kwargs.get('frequency'),
                        self.kwargs.get('summary'), self.kwargs.get('unit'))
                    print(sql)
                    #self.mysql_write(sql)
                elif self.category == 'customized':
                    sql='insert into customized_info (code,name,summary) value ("%s","%s","%s")'%(
                        self.kwargs.get('qg_code'), self.kwargs.get('name'),
                        self.kwargs.get('summary'),
                    )
                    print(sql)
                    #self.mysql_write(sql)
                logger.critical(
                    'finished loading into qgdbs for code: %s name: %s' %
                    (self.code, self.name))
            else:
                logger.critical(
                    'had problem when validation for code: %s name: %s' %
                    (self.code, self.name))
        else:

            logger.critical(
                'already had this indicator where code: %s name: %s' %
                (self.kwargs['qg_code'], self.kwargs['name']))


In [306]:
test_ticker={'qg_code':'000301.SH','name':'name','need_process':'1','category':'ticker','wind_code':'123','type':'stock','unit':'yuan'}
test_edb={'qg_code':'000301.SH','name':'name','need_process':'1','category':'edb','wind_code':'123','type':'stock','unit':'yuan'}
test_cust={'qg_code':'    000301.SH\n\t','name':'name','need_process':'1','category':'customized','wind_code':'123','type':'stock','unit':'yuan'}

In [307]:
r=QGCodeLoader(**test_cust).load()
r

06-17 16:47 logger       INFO     using username Local_Editor
06-17 16:47 logger       INFO     accessed to database <qgdbs>
06-17 16:47 logger       INFO     got a cursor
06-17 16:47 logger       INFO     Accepted read sql query "select qg_code from qg_indicator_info where qg_code like "c%.QG" order by qg_code desc limit 1"
06-17 16:47 logger       INFO     Closed connection to database <qgdbs>
06-17 16:47 logger       INFO     using username Local_Editor
06-17 16:47 logger       INFO     accessed to database <qgdbs>
06-17 16:47 logger       INFO     got a cursor
06-17 16:47 logger       INFO     Accepted read sql query "select name from qg_indicator_info where qg_code= "C00005.QG""
06-17 16:47 logger       INFO     Closed connection to database <qgdbs>
06-17 16:47 logger       CRITICAL finished loading into qgdbs for code: C00005.QG name: name


insert into qg_indicator_info (qg_code,name,category,description,need_process) value("C00005.QG","name","customized","None","1")
insert into customized_info (code,name,summary) value ("C00005.QG","name","None")


In [304]:
print(re.findall(r"(\w+\w)",'\tS002.7012\n')[0])

S002


In [108]:
d={'a':1}

In [112]:
if not d.get('a',False):
    print('no')
else:
    print('find')

find


In [247]:
if not (1):
    print(1)

In [284]:
a=' sdf123.sa\n'
type(re.findall(r"(\w+.\w+)",a)[0])

str

In [252]:
a

' sdf\n'